# kaiming-uniform-sf-init — ex1: initialize weight as Uniform(-sf, +sf) with sf = 1/sqrt(fan_in)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `kaiming-uniform-sf-init`. Running the final beacon cell reports progress against the `Init: Kaiming uniform SF init` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Init: Kaiming uniform SF init` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`kaiming-uniform-sf-init`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "kaiming-uniform-sf-init"
DD_SUBTOPIC = "Init: Kaiming uniform SF init"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Kaiming uniform with `sf = 1/sqrt(fan_in)` — quick refresher

ARENA's `Linear.__init__` uses the **simplified Kaiming uniform** form:

```
sf = 1 / sqrt(fan_in)
weight ~ Uniform(-sf, +sf)
```

where `fan_in` is the number of input units (`weight.shape[0]` for a `(in_features, out_features)` weight, or `in_channels * kernel_h * kernel_w` for a conv).

**vs the `sqrt(6/fan_in)` form.** The more general Kaiming uniform is `U(-sqrt(6/fan_in), +sqrt(6/fan_in))` — chosen so `Var(w) == 2/fan_in` (activation-preserving for ReLU). The `1/sqrt(fan_in)` form is what PyTorch's `nn.Linear` actually ships, and it's what ARENA uses. They are DIFFERENT scales — don't conflate.

Sampling recipe:
```python
sf = fan_in ** -0.5
weight = (t.rand(in_f, out_f) * 2 - 1) * sf   # uniform on (-sf, +sf)
```

Sanity: `weight.std()` ≈ `sf / sqrt(3)` (population std of `U(-sf, sf)`).

### Exercise 1 — initialize weight as Uniform(-sf, +sf) with sf = 1/sqrt(fan_in)

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the ARENA Kaiming-uniform recipe `sf = 1/sqrt(fan_in)`, `weight ~ Uniform(-sf, +sf)`, producing a tensor of the requested shape with the correct empirical spread.
> Keywords: kaiming, uniform, init, fan-in, scale-factor
> ```

**KCs targeted:** `kaiming-uniform-sf-init`, `rand-uniform-shift-scale`

Implement `kaiming_uniform_sf(in_features, out_features, generator)`. The Linear-layer weight initializer ARENA uses (and PyTorch's `nn.Linear` default):

1. `fan_in = in_features` (number of input units feeding each output neuron).
2. `sf = 1 / sqrt(fan_in)` (the scale factor).
3. Sample `(in_features, out_features)` floats uniformly on `(-sf, +sf)` using the provided `torch.Generator`.
4. Return as a `torch.Tensor` (not a MiniTensor — wrapping into `Parameter` is a separate atom).

**Distinct from the `sqrt(6/fan_in)` form.** Don't conflate. ARENA uses `1/sqrt(fan_in)`; the deeper-theory form is `sqrt(6/fan_in)` (activation-preserving for ReLU under stronger assumptions). Both are real, but the drill is specifically the ARENA / PyTorch default.

Hint: `t.rand(shape, generator=g)` is uniform on `[0, 1)`. To get `(-sf, +sf)`, do `(t.rand(shape, generator=g) * 2 - 1) * sf`.

Output: `torch.Tensor` of shape `(in_features, out_features)`.

In [ ]:
def kaiming_uniform_sf(
    in_features: int, out_features: int, generator: t.Generator
) -> Tensor:
    """Sample weight ~ Uniform(-1/sqrt(fan_in), +1/sqrt(fan_in))."""
    raise NotImplementedError()


def _test_ex1():
    import math
    # --- shape ---
    g = t.Generator().manual_seed(0)
    w = kaiming_uniform_sf(3, 5, g)
    assert isinstance(w, t.Tensor), f'expected torch.Tensor, got {type(w).__name__}'
    assert w.shape == (3, 5), f'shape: {w.shape}'

    # --- bounds: |w| <= sf for every element ---
    sf = 1.0 / math.sqrt(3)
    assert w.abs().max().item() <= sf + 1e-6, (
        f'all entries must lie in (-sf, +sf); max |w| = {w.abs().max().item()} '
        f'vs sf = {sf:.4f}'
    )

    # --- large-sample empirical check: std ~= sf / sqrt(3) ---
    g2 = t.Generator().manual_seed(1)
    fan_in = 100
    w_big = kaiming_uniform_sf(fan_in, 50_000, g2)  # 5M samples
    sf_big = 1.0 / math.sqrt(fan_in)
    assert w_big.abs().max().item() <= sf_big + 1e-6
    expected_std = sf_big / math.sqrt(3.0)
    empirical_std = w_big.std().item()
    rel_err = abs(empirical_std - expected_std) / expected_std
    assert rel_err < 0.02, (
        f'empirical std {empirical_std:.4f} too far from expected '
        f'{expected_std:.4f} (rel err {rel_err:.4f}); init may use wrong sf'
    )

    # --- mean ~= 0 (uniform on symmetric interval is zero-mean) ---
    assert abs(w_big.mean().item()) < 1e-3, (
        f'sample mean must be near 0, got {w_big.mean().item():.4f}'
    )

    # --- generator is honored: same seed → same tensor ---
    g_a = t.Generator().manual_seed(42)
    g_b = t.Generator().manual_seed(42)
    w_a = kaiming_uniform_sf(4, 7, g_a)
    w_b = kaiming_uniform_sf(4, 7, g_b)
    assert t.allclose(w_a, w_b), 'same seed must produce the same weight tensor'

    # --- DIFFERENT seed → different tensor (probabilistic sanity check) ---
    g_c = t.Generator().manual_seed(99)
    w_c = kaiming_uniform_sf(4, 7, g_c)
    assert not t.allclose(w_a, w_c), 'different seed should produce different tensor'

    # --- scale shrinks with fan_in: sf for fan_in=400 is half of sf for fan_in=100 ---
    g_big1 = t.Generator().manual_seed(7)
    g_big2 = t.Generator().manual_seed(7)
    w_100 = kaiming_uniform_sf(100, 1000, g_big1)
    w_400 = kaiming_uniform_sf(400, 1000, g_big2)
    ratio = w_400.abs().max().item() / w_100.abs().max().item()
    # sf_400 / sf_100 = sqrt(100/400) = 0.5
    assert 0.35 < ratio < 0.65, (
        f'sf must scale as 1/sqrt(fan_in); got max-abs ratio {ratio:.3f}, expected ~0.5'
    )
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def kaiming_uniform_sf(
    in_features: int, out_features: int, generator: t.Generator
) -> Tensor:
    # sf = 1 / sqrt(fan_in)  — ARENA / PyTorch nn.Linear default
    sf = in_features ** -0.5
    # Uniform(-sf, +sf) = (Uniform(0, 1) * 2 - 1) * sf
    raw = t.rand(in_features, out_features, generator=generator)
    return (raw * 2 - 1) * sf
```

**The two Kaiming forms are NOT the same.** ARENA / PyTorch use `U(-1/sqrt(fan_in), +1/sqrt(fan_in))`. The 'activation-preserving' form is `U(-sqrt(6/fan_in), +sqrt(6/fan_in))` — chosen so `Var(w) = 2/fan_in`. With the ARENA form, `Var(w) = sf^2 / 3 = 1/(3 * fan_in)` — smaller by a factor of 6. Both initialize networks that train; they're calibrated for different objective functions.

**Why the empirical std ≈ sf / sqrt(3).** The population std of `Uniform(-a, +a)` is `a / sqrt(3)` (variance = `a^2 / 3`). With `a = sf`, that's `sf / sqrt(3)`. The test asserts this to within 2% relative error using 5M samples — sufficient to catch the wrong-form init (which would land at `sf * sqrt(2)`).

**Generator threading.** Passing a `torch.Generator` instead of calling `t.rand()` globally lets the caller seed reproducibly — essential for testable inits, deterministic experiments, and regression tests that compare two networks' weights.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()